# Adult Income Data Preparation

This notebook turns the findings from the data-quality diagnosis into two prepared modeling views for the Adult Income dataset.

Goals of this notebook:
- keep the preparation pipeline explicit and reviewable
- stay conservative on data removal
- build one tree-first view and one linear-model comparison view
- preserve official metadata and source references for every feature


In [1]:
from collections import OrderedDict
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)


In [2]:
def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data" / "adult-pmr3508").exists():
            return candidate
    raise FileNotFoundError("Could not find data/adult-pmr3508 from the current working directory.")


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "adult-pmr3508"
TRAIN_PATH = DATA_DIR / "train_data.csv"
TEST_PATH = DATA_DIR / "test_data.csv"
EXTRA_PATH = DATA_DIR / "Extra-file-from-UCI.txt"

SOURCE_URLS = {
    "kaggle": "https://www.kaggle.com/competitions/adult-pmr3508/data",
    "uci": "https://archive.ics.uci.edu/dataset/2/adult",
    "extra": "data/adult-pmr3508/Extra-file-from-UCI.txt",
    "catboost_categorical": "https://catboost.ai/docs/en/features/categorical-features.html",
    "catboost_missing": "https://catboost.ai/docs/en/concepts/algorithm-missing-values-processing.html",
    "xgboost_categorical": "https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html",
    "sklearn_ohe": "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html",
    "sklearn_robust_scaler": "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html",
    "sklearn_quantile": "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html",
    "sklearn_target_encoder": "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html",
    "grinsztajn_2022": "https://arxiv.org/abs/2207.08815",
    "mcelfresh_2023": "https://arxiv.org/abs/2305.02997"
}

ID_COL = "Id"
TARGET_COL = "income"
MISSING_SENTINEL_COLUMNS = ["workclass", "occupation", "native.country"]

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
uci_text = EXTRA_PATH.read_text()

feature_columns_raw = [column for column in train_raw.columns if column not in [ID_COL, TARGET_COL]]
raw_numeric_columns = train_raw[feature_columns_raw].select_dtypes(include=np.number).columns.tolist()
raw_categorical_columns = train_raw[feature_columns_raw].select_dtypes(include=["object", "string"]).columns.tolist()

target_map = {"<=50K": 0, ">50K": 1}
y_train = train_raw[TARGET_COL].map(target_map).astype(int)

NORMALIZED_UCI_NAME_MAP = {
    "education-num": "education.num",
    "marital-status": "marital.status",
    "capital-gain": "capital.gain",
    "capital-loss": "capital.loss",
    "hours-per-week": "hours.per.week",
    "native-country": "native.country"
}


def normalize_uci_name(name: str) -> str:
    normalized = name.strip().lower()
    return NORMALIZED_UCI_NAME_MAP.get(normalized, normalized)



def parse_uci_variable_metadata(text: str) -> dict:
    metadata = {}
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("|") or ":" not in stripped:
            continue
        name, description = stripped.split(":", 1)
        normalized_name = normalize_uci_name(name)
        if normalized_name not in set(train_raw.columns):
            continue
        metadata[normalized_name] = description.strip().rstrip(".")
    return metadata


uci_variable_metadata = parse_uci_variable_metadata(uci_text)

DETAILED_DESCRIPTIONS = {
    "Id": "Competition row identifier supplied by Kaggle for submission alignment; not part of the original UCI feature set.",
    "age": "Age in years for the individual record.",
    "workclass": "Employment class or type of employer, such as private sector, self-employment, or government service.",
    "fnlwgt": "Final person weight from the Current Population Survey; a survey-weight variable used to align records with population estimates.",
    "education": "Highest education category recorded for the individual.",
    "education.num": "Ordinal numeric encoding of the education ladder used in the Adult dataset.",
    "marital.status": "Current marital-status category for the individual.",
    "occupation": "Occupation or job-family category.",
    "relationship": "Relationship role of the person within the household.",
    "race": "Race category recorded in the census-derived data.",
    "sex": "Sex category recorded in the census-derived data.",
    "capital.gain": "Capital-gain amount recorded for the individual.",
    "capital.loss": "Capital-loss amount recorded for the individual.",
    "hours.per.week": "Usual hours worked per week.",
    "native.country": "Country-of-origin / native-country category recorded for the individual.",
    "income": "Binary target indicating whether annual income exceeds 50K."
}

MODELING_STATUS = {
    "Id": "drop_from_model_keep_for_traceability",
    "education": "derive_then_drop",
    "income": "target"
}

FEATURE_NOTES = {
    "education": "`education` is redundant with `education.num`; the prepared views keep `education.num` as canonical and derive a grouped education feature.",
    "education.num": "Canonical fine-grained ordinal education signal retained in prepared outputs.",
    "fnlwgt": "Keep as a feature, but do not use it as an impossibility filter or as a sample-weight default in this notebook.",
    "workclass": "Contains official unknown values encoded as `?` in the raw files.",
    "occupation": "Contains official unknown values encoded as `?` in the raw files.",
    "native.country": "Contains official unknown values encoded as `?`; missingness is informative enough to preserve explicitly."
}


def allowed_values_or_range(column: str) -> str:
    if column == ID_COL:
        return "Competition identifier; observed as unique integer key in train and test."
    if column == TARGET_COL:
        return ">50K, <=50K"
    official = uci_variable_metadata.get(column)
    if official and official.lower().startswith("continuous"):
        train_min = train_raw[column].min()
        train_max = train_raw[column].max()
        return f"continuous (official); observed train range: {train_min} to {train_max}"
    if official:
        return official
    return "Not listed in the UCI variable table."


feature_catalog = OrderedDict()
for column in train_raw.columns:
    feature_catalog[column] = {
        "role": "feature" if column not in [ID_COL, TARGET_COL] else ("identifier" if column == ID_COL else "target"),
        "dtype_family": (
            "numeric" if column in raw_numeric_columns else
            ("categorical" if column in raw_categorical_columns else ("identifier" if column == ID_COL else "target"))
        ),
        "official_description": DETAILED_DESCRIPTIONS[column],
        "allowed_values_or_range": allowed_values_or_range(column),
        "present_in_train": True,
        "present_in_test": column in test_raw.columns,
        "modeling_status": MODELING_STATUS.get(column, "keep"),
        "notes": (
            "Prof-school is professional school; Masters is completed master's; Doctorate is completed doctorate / PhD."
            if column in ["education", "education.num"] else FEATURE_NOTES.get(column, "")
        ),
        "sources": [SOURCE_URLS["kaggle"], SOURCE_URLS["uci"], SOURCE_URLS["extra"]]
    }

feature_catalog_df = pd.DataFrame(feature_catalog).T.reset_index(names="feature")

print(f"Project root: {PROJECT_ROOT}")
print(f"Train shape: {train_raw.shape}")
print(f"Test shape: {test_raw.shape}")


Project root: /home/matheuscm/classifier-adults-dataset
Train shape: (32560, 16)
Test shape: (16280, 15)


## Feature Catalog

The preparation notebook starts with a source-backed feature catalog. Each field below combines competition context from Kaggle with variable semantics from UCI and the local `Extra-file-from-UCI.txt` companion file.


In [3]:
pprint(feature_catalog, sort_dicts=False)
display(feature_catalog_df)


OrderedDict([('Id',
              {'role': 'identifier',
               'dtype_family': 'identifier',
               'official_description': 'Competition row identifier supplied by '
                                       'Kaggle for submission alignment; not '
                                       'part of the original UCI feature set.',
               'allowed_values_or_range': 'Competition identifier; observed as '
                                          'unique integer key in train and '
                                          'test.',
               'present_in_train': True,
               'present_in_test': True,
               'modeling_status': 'drop_from_model_keep_for_traceability',
               'notes': '',
               'sources': ['https://www.kaggle.com/competitions/adult-pmr3508/data',
                           'https://archive.ics.uci.edu/dataset/2/adult',
                           'data/adult-pmr3508/Extra-file-from-UCI.txt']}),
             ('age',
         

,feature,role,dtype_family,official_description,allowed_values_or_range,present_in_train,present_in_test,modeling_status,notes,sources
0,Id,identifier,identifier,Competition row identifier supplied by Kaggle for submission alignment; not part of the original UCI feature set.,Competition identifier; observed as unique integer key in train and test.,True,True,drop_from_model_keep_for_traceability,,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
1,age,feature,numeric,Age in years for the individual record.,continuous (official); observed train range: 17 to 90,True,True,keep,,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
2,workclass,feature,categorical,"Employment class or type of employer, such as private sector, self-employment, or government service.","Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked",True,True,keep,Contains official unknown values encoded as `?` in the raw files.,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
3,fnlwgt,feature,numeric,Final person weight from the Current Population Survey; a survey-weight variable used to align records with population estimates.,continuous (official); observed train range: 12285 to 1484705,True,True,keep,"Keep as a feature, but do not use it as an impossibility filter or as a sample-weight default in this notebook.","[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
4,education,feature,categorical,Highest education category recorded for the individual.,"Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool",True,True,derive_then_drop,Prof-school is professional school; Masters is completed master's; Doctorate is completed doctorate / PhD.,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
5,education.num,feature,numeric,Ordinal numeric encoding of the education ladder used in the Adult dataset.,continuous (official); observed train range: 1 to 16,True,True,keep,Prof-school is professional school; Masters is completed master's; Doctorate is completed doctorate / PhD.,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
6,marital.status,feature,categorical,Current marital-status category for the individual.,"Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse",True,True,keep,,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
7,occupation,feature,categorical,Occupation or job-family category.,"Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Tran...",True,True,keep,Contains official unknown values encoded as `?` in the raw files.,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
8,relationship,feature,categorical,Relationship role of the person within the household.,"Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried",True,True,keep,,"[https://www.kaggle.com/competitions/adult-pmr3508/data, https://archive.ics.uci.edu/dataset/2/adult, data/adult-pmr3508/Extra-file-from-UCI.txt]"
9,race,feature,categorical,Race category recorded in the census-derived data.,"White, Asian-Pac-Islander, Amer-Indi

## Key Quality Findings Carried Into Preparation

These checks reproduce the main preparation-relevant findings from the diagnosis notebook so that the prep notebook remains self-sufficient.


In [4]:
def question_mark_count(series: pd.Series) -> int:
    if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
        return 0
    return int(series.astype("string").str.strip().eq("?").sum())


missing_like_records = []
for split_name, frame in [("train", train_raw), ("test", test_raw)]:
    for column in frame.columns:
        question_marks = question_mark_count(frame[column])
        actual_nulls = int(frame[column].isna().sum())
        if question_marks or actual_nulls:
            missing_like_records.append(
                {
                    "split": split_name,
                    "column": column,
                    "actual_nulls": actual_nulls,
                    "question_marks": question_marks,
                    "missing_like_count": actual_nulls + question_marks,
                    "missing_like_pct": (actual_nulls + question_marks) / len(frame)
                }
            )
missing_like_df = pd.DataFrame(missing_like_records)
if not missing_like_df.empty:
    missing_like_df["missing_like_pct"] = missing_like_df["missing_like_pct"].map(lambda value: f"{value:.2%}")

exact_duplicate_mask = train_raw.duplicated(subset=[column for column in train_raw.columns if column != ID_COL], keep="first")
exact_duplicate_count = int(exact_duplicate_mask.sum())

feature_only_columns = [column for column in train_raw.columns if column not in [ID_COL, TARGET_COL]]
conflict_groups_df = (
    train_raw.groupby(feature_only_columns, dropna=False)[TARGET_COL]
    .nunique()
    .reset_index(name="target_cardinality")
    .query("target_cardinality > 1")
)
conflicting_rows_df = train_raw.merge(conflict_groups_df[feature_only_columns], on=feature_only_columns, how="inner")

hours_extremes_df = pd.DataFrame(
    [
        {"threshold": ">= 60", "count": int((train_raw["hours.per.week"] >= 60).sum())},
        {"threshold": ">= 80", "count": int((train_raw["hours.per.week"] >= 80).sum())},
        {"threshold": ">= 90", "count": int((train_raw["hours.per.week"] >= 90).sum())}
    ]
)
hours_extremes_df["pct_of_train"] = (hours_extremes_df["count"] / len(train_raw)).map(lambda value: f"{value:.2%}")

zero_inflation_df = pd.DataFrame(
    [
        {
            "column": column,
            "zero_count": int((train_raw[column] == 0).sum()),
            "zero_pct": f"{(train_raw[column] == 0).mean():.2%}"
        }
        for column in ["capital.gain", "capital.loss"]
    ]
)

diagnostic_summary_df = pd.DataFrame(
    [
        {"finding": "train rows", "value": len(train_raw)},
        {"finding": "test rows", "value": len(test_raw)},
        {"finding": "exact duplicate train rows (same predictors and label, excluding Id)", "value": exact_duplicate_count},
        {"finding": "conflicting duplicate feature groups", "value": len(conflict_groups_df)},
        {"finding": "max hours.per.week", "value": int(train_raw["hours.per.week"].max())},
        {"finding": "max capital.gain", "value": int(train_raw["capital.gain"].max())},
        {"finding": "max capital.loss", "value": int(train_raw["capital.loss"].max())}
    ]
)

display(diagnostic_summary_df)
display(missing_like_df.sort_values(["split", "missing_like_count"], ascending=[True, False]))
display(hours_extremes_df)
display(zero_inflation_df)
display(conflicting_rows_df[[ID_COL, "age", "workclass", "fnlwgt", "education", "education.num", "marital.status", "occupation", "relationship", "race", "sex", "capital.gain", "capital.loss", "hours.per.week", "native.country", TARGET_COL]])


,finding,value
0,train rows,32560
1,test rows,16280
2,"exact duplicate train rows (same predictors and label, excluding Id)",24
3,conflicting duplicate feature groups,1
4,max hours.per.week,99
5,max capital.gain,99999
6,max capital.loss,4356


,split,column,actual_nulls,question_marks,missing_like_count,missing_like_pct
4,test,occupation,0,966,966,5.93%
3,test,workclass,0,963,963,5.92%
5,test,native.country,0,274,274,1.68%
1,train,occupation,0,1843,1843,5.66%
0,train,workclass,0,1836,1836,5.64%
2,train,native.country,0,583,583,1.79%


,threshold,count,pct_of_train
0,>= 60,2585,7.94%
1,>= 80,341,1.05%
2,>= 90,139,0.43%


,column,zero_count,zero_pct
0,capital.gain,29849,91.67%
1,capital.loss,31041,95.33%


,Id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,44434,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
1,48266,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K


## Why This Notebook Uses a Tree-First, Dual-Track Preparation Strategy

Preparation choices are guided by the observed diagnostics and by official or benchmark-oriented references:

- Historical Adult baselines in `Extra-file-from-UCI.txt` show strong performance from tree-style methods such as NBTree and C4.5-family variants, which makes a tree-first branch a sensible default for this dataset.
- Recent tabular benchmark literature still finds gradient-boosted tree families very competitive on average for medium-scale structured data. References: Grinsztajn et al. (2022) and McElfresh et al. (2023 / TabZilla).
- Official CatBoost and XGBoost documentation explicitly support missing-value handling and categorical workflows well suited to mixed-type tabular data.
- Official scikit-learn preprocessing documentation supports two complementary choices for the comparison branch:
  - `RobustScaler` for scaled continuous inputs that are less sensitive to outliers than standard scaling
  - `OneHotEncoder(handle_unknown="infrequent_if_exist")` to stabilize rare-category handling in linear models
- `QuantileTransformer` is a viable alternative for strongly skewed numeric variables, but this notebook keeps the default linear branch more interpretable by using explicit `log1p` transforms plus `RobustScaler`.
- The education grouping preserves educational ordering while separating terminal advanced degrees:
  - `Prof-school` remains distinct
  - `Masters` stays separate from `Doctorate`
  - `education.num` remains the fine-grained ordinal signal

Reference links used in this notebook:
- UCI Adult dataset: https://archive.ics.uci.edu/dataset/2/adult
- Kaggle competition data page: https://www.kaggle.com/competitions/adult-pmr3508/data
- CatBoost categorical features: https://catboost.ai/docs/en/features/categorical-features.html
- CatBoost missing values: https://catboost.ai/docs/en/concepts/algorithm-missing-values-processing.html
- XGBoost categorical data: https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html
- scikit-learn OneHotEncoder: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
- scikit-learn RobustScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html
- scikit-learn QuantileTransformer: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html
- scikit-learn TargetEncoder: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html
- Grinsztajn et al. (2022): https://arxiv.org/abs/2207.08815
- McElfresh et al. (2023 / TabZilla): https://arxiv.org/abs/2305.02997


## Shared Base Cleaning Layer

This layer standardizes missing-value handling, carries duplicate diagnostics forward, and removes only exact training duplicates with identical predictors and identical target label.


In [5]:
def replace_missing_sentinels(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in MISSING_SENTINEL_COLUMNS:
        cleaned[column] = cleaned[column].replace("?", pd.NA)
    cleaned["workclass_missing"] = cleaned["workclass"].isna().astype(int)
    cleaned["occupation_missing"] = cleaned["occupation"].isna().astype(int)
    cleaned["native_country_missing"] = cleaned["native.country"].isna().astype(int)
    return cleaned


train_base = replace_missing_sentinels(train_raw)
test_base = replace_missing_sentinels(test_raw)

train_trace_flags = pd.DataFrame({
    ID_COL: train_raw[ID_COL],
    "was_exact_duplicate_removed_candidate": exact_duplicate_mask.astype(bool)
})

conflict_keys = conflict_groups_df[feature_only_columns].copy()
if conflict_keys.empty:
    train_trace_flags["is_conflicting_duplicate_features"] = False
else:
    conflict_keys["is_conflicting_duplicate_features"] = True
    conflicting_flag_rows = train_raw[[ID_COL] + feature_only_columns].merge(
        conflict_keys,
        on=feature_only_columns,
        how="left"
    )[[ID_COL, "is_conflicting_duplicate_features"]]
    conflicting_flag_rows["is_conflicting_duplicate_features"] = conflicting_flag_rows["is_conflicting_duplicate_features"].fillna(False).astype(bool)
    train_trace_flags = train_trace_flags.merge(conflicting_flag_rows, on=ID_COL, how="left")
    train_trace_flags["is_conflicting_duplicate_features"] = train_trace_flags["is_conflicting_duplicate_features"].fillna(False).astype(bool)

train_base = train_base.merge(train_trace_flags, on=ID_COL, how="left")
train_model = train_base.loc[~train_base["was_exact_duplicate_removed_candidate"]].copy()
test_model = test_base.copy()

removed_exact_duplicates_df = train_base.loc[train_base["was_exact_duplicate_removed_candidate"]].copy()
train_sensitivity_no_conflicts = train_model.loc[~train_model["is_conflicting_duplicate_features"]].copy()

base_cleaning_summary_df = pd.DataFrame(
    [
        {"dataset": "train_raw", "rows": len(train_raw)},
        {"dataset": "removed_exact_duplicates", "rows": len(removed_exact_duplicates_df)},
        {"dataset": "train_model_after_exact_dedup", "rows": len(train_model)},
        {"dataset": "train_sensitivity_no_conflicts", "rows": len(train_sensitivity_no_conflicts)},
        {"dataset": "test_model", "rows": len(test_model)}
    ]
)

display(base_cleaning_summary_df)
display(removed_exact_duplicates_df.head(10))
display(train_model.loc[train_model["is_conflicting_duplicate_features"], [ID_COL, "age", "workclass", "fnlwgt", "education", "education.num", "marital.status", "occupation", "relationship", "race", "sex", "capital.gain", "capital.loss", "hours.per.week", "native.country", TARGET_COL, "is_conflicting_duplicate_features"]])


,dataset,rows
0,train_raw,32560
1,removed_exact_duplicates,24
2,train_model_after_exact_dedup,32536
3,train_sensitivity_no_conflicts,32534
4,test_model,16280


,Id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,workclass_missing,occupation_missing,native_country_missing,was_exact_duplicate_removed_candidate,is_conflicting_duplicate_features
4920,21200,21,Private,250051,Some-college,10,Never-married,Prof-specialty,Own-child,White,Female,0,0,10,United-States,<=50K,0,0,0,True,False
9783,26063,46,Private,173243,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K,0,0,0,True,False
11171,27451,20,Private,107658,Some-college,10,Never-married,Tech-support,Not-in-family,White,Female,0,0,10,United-States,<=50K,0,0,0,True,False
11401,27681,38,Private,207202,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,48,United-States,>50K,0,0,0,True,False
15454,31734,19,Private,251579,Some-college,10,Never-married,Other-service,Own-child,White,Male,0,0,14,United-States,<=50K,0,0,0,True,False
18298,34578,25,Private,195994,1st-4th,2,Never-married,Priv-house-serv,Not-in-family,White,Female,0,0,40,Guatemala,<=50K,0,0,0,True,False
20971,37251,35,Private,379959,HS-grad,9,Divorced,Other-service,Not-in-family,White,Female,0,0,40,United-States,<=50K,0,0,0,True,False
21324,37604,23,Private,240137,5th-6th,3,Never-married,Handlers-cleaners,Not-in-family,White,Male,0,0,55,Mexico,<=50K,0,0,0,True,False
21936,38216,25,Private,195994,1st-4th,2,Never-married,Priv-house-serv,Not-in-family,White,Female,0,0,40,Guatemala,<=50K,0,0,0,True,False
22524,38804,27,Private,255582,HS-grad,9,Never-married,Machine-op-inspct,Not-in-family,White,Female,0,0,40,United-States,<=50K,0,0,0,True,False


,Id,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,is_conflicting_duplicate_features
28154,44434,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K,True
31986,48266,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K,True


## Feature Engineering

The engineered features below respond directly to the diagnosis notebook:
- skew-aware numeric companions instead of row-level outlier removal
- explicit missingness indicators
- grouped education signal with separate `masters` and `doctorate`
- household and workload features suggested by strong interaction patterns in the diagnosis step


In [7]:
EDUCATION_LEVEL_GROUP_MAP = {
    "Preschool": "dropout",
    "1st-4th": "dropout",
    "5th-6th": "dropout",
    "7th-8th": "dropout",
    "9th": "dropout",
    "10th": "dropout",
    "11th": "dropout",
    "12th": "dropout",
    "HS-grad": "hs_grad",
    "Some-college": "some_college",
    "Assoc-acdm": "associate",
    "Assoc-voc": "associate",
    "Bachelors": "bachelors",
    "Prof-school": "professional_school",
    "Masters": "masters",
    "Doctorate": "doctorate"
}

AGE_BAND_BINS = [17, 25, 35, 45, 55, 65, np.inf]
AGE_BAND_LABELS = ["17_24", "25_34", "35_44", "45_54", "55_64", "65_plus"]


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    engineered = frame.copy()
    engineered["education_level_group"] = engineered["education"].map(EDUCATION_LEVEL_GROUP_MAP)
    engineered["log1p_fnlwgt"] = np.log1p(engineered["fnlwgt"])
    engineered["capital_gain_positive"] = (engineered["capital.gain"] > 0).astype(int)
    engineered["capital_loss_positive"] = (engineered["capital.loss"] > 0).astype(int)
    engineered["log1p_capital_gain"] = np.log1p(engineered["capital.gain"])
    engineered["log1p_capital_loss"] = np.log1p(engineered["capital.loss"])
    engineered["hours_ge_50"] = (engineered["hours.per.week"] >= 50).astype(int)
    engineered["hours_ge_60"] = (engineered["hours.per.week"] >= 60).astype(int)
    engineered["hours_ge_80"] = (engineered["hours.per.week"] >= 80).astype(int)
    engineered["age_band"] = pd.cut(
        engineered["age"],
        bins=AGE_BAND_BINS,
        labels=AGE_BAND_LABELS,
        right=False,
        include_lowest=True
    ).astype("string")
    marital = engineered["marital.status"].astype("string").fillna("__MISSING__")
    relationship = engineered["relationship"].astype("string").fillna("__MISSING__")
    engineered["family_role_combo"] = marital + "__" + relationship
    return engineered


train_features = engineer_features(train_model)
test_features = engineer_features(test_model)

education_mapping_df = (
    pd.DataFrame({
        "education": list(EDUCATION_LEVEL_GROUP_MAP.keys()),
        "education_level_group": list(EDUCATION_LEVEL_GROUP_MAP.values())
    })
    .merge(
        train_raw[["education", "education.num"]].drop_duplicates().sort_values(["education.num", "education"]),
        on="education",
        how="left"
    )
    .sort_values(["education.num", "education"])
)

assert education_mapping_df["education_level_group"].notna().all(), "Every education value must map to a grouped level."
assert education_mapping_df["education"].nunique() == train_raw["education"].nunique(), "The mapping table must cover all observed education levels."

display(education_mapping_df)
display(train_features[["education", "education.num", "education_level_group", "age", "age_band", "hours.per.week", "hours_ge_50", "hours_ge_60", "hours_ge_80", "family_role_combo"]].head(10))


,education,education_level_group,education.num
0,Preschool,dropout,1
1,1st-4th,dropout,2
2,5th-6th,dropout,3
3,7th-8th,dropout,4
4,9th,dropout,5
5,10th,dropout,6
6,11th,dropout,7
7,12th,dropout,8
8,HS-grad,hs_grad,9
9,Some-college,some_college,10


,education,education.num,education_level_group,age,age_band,hours.per.week,hours_ge_50,hours_ge_60,hours_ge_80,family_role_combo
0,Some-college,10,some_college,34,25_34,44,0,0,0,Divorced__Own-child
1,10th,6,dropout,58,55_64,40,0,0,0,Married-civ-spouse__Husband
2,Some-college,10,some_college,25,25_34,42,0,0,0,Never-married__Not-in-family
3,Some-college,10,some_college,24,17_24,40,0,0,0,Divorced__Not-in-family
4,HS-grad,9,hs_grad,57,55_64,60,1,1,0,Married-civ-spouse__Husband
5,HS-grad,9,hs_grad,57,55_64,38,0,0,0,Married-civ-spouse__Wife
6,Some-college,10,some_college,21,17_24,35,0,0,0,Never-married__Own-child
7,Bachelors,13,bachelors,25,25_34,40,0,0,0,Never-married__Not-in-family
8,HS-grad,9,hs_grad,53,45_54,30,0,0,0,Widowed__Not-in-family
9,HS-grad,9,hs_grad,30,25_34,40,0,0,0,Never-married__Own-child


## Tree-Ready View

This branch is the primary prepared output. It preserves categorical variables as categorical features, keeps rare categories intact, and avoids scaling.


In [8]:
TREE_CATEGORICAL_COLUMNS = [
    "workclass",
    "marital.status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native.country",
    "education_level_group",
    "age_band",
    "family_role_combo"
]

TREE_NUMERIC_COLUMNS = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week",
    "log1p_fnlwgt",
    "capital_gain_positive",
    "capital_loss_positive",
    "log1p_capital_gain",
    "log1p_capital_loss",
    "hours_ge_50",
    "hours_ge_60",
    "hours_ge_80",
    "workclass_missing",
    "occupation_missing",
    "native_country_missing"
]

TREE_FEATURE_COLUMNS = TREE_NUMERIC_COLUMNS + TREE_CATEGORICAL_COLUMNS


def make_tree_ready(frame: pd.DataFrame) -> pd.DataFrame:
    tree_ready = frame[TREE_FEATURE_COLUMNS].copy()
    for column in TREE_CATEGORICAL_COLUMNS:
        tree_ready[column] = tree_ready[column].astype("string").fillna("__MISSING__").astype("category")
    return tree_ready


tree_ready_train_X = make_tree_ready(train_features)
tree_ready_test_X = make_tree_ready(test_features)
tree_ready_train_y = train_features[TARGET_COL].map(target_map).astype(int)

tree_ready_summary_df = pd.DataFrame(
    [
        {"split": "train", "rows": len(tree_ready_train_X), "columns": tree_ready_train_X.shape[1]},
        {"split": "test", "rows": len(tree_ready_test_X), "columns": tree_ready_test_X.shape[1]}
    ]
)

tree_ready_dtypes_df = pd.DataFrame(
    {
        "feature": tree_ready_train_X.columns,
        "dtype": tree_ready_train_X.dtypes.astype(str).values
    }
)

assert list(tree_ready_train_X.columns) == list(tree_ready_test_X.columns), "Tree-ready train/test columns must match."

display(tree_ready_summary_df)
display(tree_ready_dtypes_df)
display(tree_ready_train_X.head())


,split,rows,columns
0,train,32536,27
1,test,16280,27


,feature,dtype
0,age,int64
1,fnlwgt,int64
2,education.num,int64
3,capital.gain,int64
4,capital.loss,int64
5,hours.per.week,int64
6,log1p_fnlwgt,float64
7,capital_gain_positive,int64
8,capital_loss_positive,int64
9,log1p_capital_gain,float64


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,log1p_fnlwgt,capital_gain_positive,capital_loss_positive,log1p_capital_gain,log1p_capital_loss,hours_ge_50,hours_ge_60,hours_ge_80,workclass_missing,occupation_missing,native_country_missing,workclass,marital.status,occupation,relationship,race,sex,native.country,education_level_group,age_band,family_role_combo
0,34,204991,10,0,0,44,12.230726,0,0,0.000000,0.0,0,0,0,0,0,0,Private,Divorced,Exec-managerial,Own-child,White,Male,United-States,some_college,25_34,Divorced__Own-child
1,58,310085,6,0,0,40,12.644605,0,0,0.000000,0.0,0,0,0,0,0,0,Local-gov,Married-civ-spouse,Transport-moving,Husband,White,Male,United-States,dropout,55_64,Married-civ-spouse__Husband
2,25,146117,10,0,0,42,11.892170,0,0,0.000000,0.0,0,0,0,0,0,0,Private,Never-married,Machine-op-inspct,Not-in-family,White,Male,United-States,some_college,25_34,Never-married__Not-in-family
3,24,138938,10,0,0,40,11.841790,0,0,0.000000,0.0,0,0,0,0,0,0,Private,Divorced,Adm-clerical,Not-in-family,White,Female,United-States,some_college,17_24,Divorced__Not-in-family
4,57,258883,9,5178,0,60,12.464135,1,0,8.552367,0.0,1,1,0,0,0,0,Self-emp-inc,Married-civ-spouse,Transport-moving,Husband,White,Male,Hungary,hs_grad,55_64,Married-civ-spouse__Husband


## Linear-Ready View

This branch uses one-hot encoding with infrequent-category handling plus `RobustScaler` for selected continuous inputs. The raw skewed versions of `fnlwgt`, `capital.gain`, and `capital.loss` are intentionally excluded here in favor of transformed companions.


In [9]:
LINEAR_SCALED_NUMERIC_COLUMNS = [
    "age",
    "education.num",
    "hours.per.week",
    "log1p_fnlwgt",
    "log1p_capital_gain",
    "log1p_capital_loss"
]

LINEAR_BINARY_COLUMNS = [
    "capital_gain_positive",
    "capital_loss_positive",
    "hours_ge_50",
    "hours_ge_60",
    "hours_ge_80",
    "workclass_missing",
    "occupation_missing",
    "native_country_missing"
]

LINEAR_CATEGORICAL_COLUMNS = TREE_CATEGORICAL_COLUMNS.copy()


def make_linear_source(frame: pd.DataFrame) -> pd.DataFrame:
    linear_source = frame[LINEAR_SCALED_NUMERIC_COLUMNS + LINEAR_BINARY_COLUMNS + LINEAR_CATEGORICAL_COLUMNS].copy()
    for column in LINEAR_CATEGORICAL_COLUMNS:
        linear_source[column] = linear_source[column].astype("string").fillna("__MISSING__")
    return linear_source


linear_train_source = make_linear_source(train_features)
linear_test_source = make_linear_source(test_features)

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("scaled_numeric", RobustScaler(), LINEAR_SCALED_NUMERIC_COLUMNS),
        ("binary_flags", "passthrough", LINEAR_BINARY_COLUMNS),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=20,
                sparse_output=False
            ),
            LINEAR_CATEGORICAL_COLUMNS
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)
linear_preprocessor.set_output(transform="pandas")

linear_ready_train_X = linear_preprocessor.fit_transform(linear_train_source)
linear_ready_test_X = linear_preprocessor.transform(linear_test_source)
linear_ready_train_y = tree_ready_train_y.copy()

linear_ready_summary_df = pd.DataFrame(
    [
        {"split": "train", "rows": len(linear_ready_train_X), "columns": linear_ready_train_X.shape[1]},
        {"split": "test", "rows": len(linear_ready_test_X), "columns": linear_ready_test_X.shape[1]}
    ]
)

linear_schema_check_df = pd.DataFrame(
    {
        "train_test_same_columns": [list(linear_ready_train_X.columns) == list(linear_ready_test_X.columns)],
        "encoded_feature_count": [linear_ready_train_X.shape[1]]
    }
)

assert list(linear_ready_train_X.columns) == list(linear_ready_test_X.columns), "Linear-ready train/test columns must match."

display(linear_ready_summary_df)
display(linear_schema_check_df)
display(linear_ready_train_X.head())


,split,rows,columns
0,train,32536,128
1,test,16280,128


,train_test_same_columns,encoded_feature_count
0,True,128


,age,education.num,hours.per.week,log1p_fnlwgt,log1p_capital_gain,log1p_capital_loss,capital_gain_positive,capital_loss_positive,hours_ge_50,hours_ge_60,hours_ge_80,workclass_missing,occupation_missing,native_country_missing,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass___MISSING__,workclass_infrequent_sklearn,marital.status_Divorced,marital.status_Married-AF-spouse,marital.status_Married-civ-spouse,marital.status_Married-spouse-absent,marital.status_Never-married,marital.status_Separated,marital.status_Widowed,occupation_Adm-clerical,occupation_Craft-repair,occupation_Exec-managerial,occupation_Farming-fishing,occupation_Handlers-cleaners,occupation_Machine-op-inspct,occupation_Other-service,occupation_Priv-house-serv,occupation_Prof-specialty,occupation_Protective-serv,occupation_Sales,occupation_Tech-support,occupation_Transport-moving,occupation___MISSING__,occupation_infrequent_sklearn,relationship_Husband,relationship_Not-in-family,relationship_Other-relative,relationship_Own-child,relationship_Unmarried,relationship_Wife,race_Amer-Indian-Eskimo,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Female,sex_Male,native.country_Canada,native.country_China,native.country_Columbia,native.country_Cuba,native.country_Dominican-Republic,native.country_Ecuador,native.country_El-Salvador,native.country_England,native.country_France,native.country_Germany,native.country_Greece,native.country_Guatemala,native.country_Haiti,native.country_Hong,native.country_India,native.country_Iran,native.country_Ireland,native.country_Italy,native.country_Jamaica,native.country_Japan,native.country_Mexico,native.country_Nicaragua,native.country_Peru,native.country_Philippines,native.country_Poland,native.country_Portugal,native.country_Puerto-Rico,native.country_South,native.country_Taiwan,native.country_United-States,native.country_Vietnam,native.country___MISSING__,native.country_infrequent_sklearn,education_level_group_associate,education_level_group_bachelors,education_level_group_doctorate,education_level_group_dropout,education_level_group_hs_grad,education_level_group_masters,education_level_group_professional_school,education_level_group_some_college,age_band_17_24,age_band_25_34,age_band_35_44,age_band_45_54,age_band_55_64,age_band_65_plus,family_role_combo_Divorced__Not-in-family,family_role_combo_Divorced__Other-relative,family_role_combo_Divorced__Own-child,family_role_combo_Divorced__Unmarried,family_role_combo_Married-civ-spouse__Husband,family_role_combo_Married-civ-spouse__Other-relative,family_role_combo_Married-civ-spouse__Own-child,family_role_combo_Married-civ-spouse__Wife,family_role_combo_Married-spouse-absent__Not-in-family,family_role_combo_Married-spouse-absent__Other-relative,family_role_combo_Married-spouse-absent__Own-child,family_role_combo_Married-spouse-absent__Unmarried,family_role_combo_Never-married__Not-in-family,family_role_combo_Never-married__Other-relative,family_role_combo_Never-married__Own-child,family_role_combo_Never-married__Unmarried,family_role_combo_Separated__Not-in-family,family_role_combo_Separated__Other-relative,family_role_combo_Separated__Own-child,family_role_combo_Separated__Unmarried,family_role_combo_Widowed__Not-in-family,family_role_combo_Widowed__Other-relative,family_role_combo_Widowed__Unmarried,family_role_combo_infrequent_sklearn
0,-0.15,0.000000,0.8,0.199183,0.000000,0.0,0,0,0,0,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.05,-1.333333,0.0,0.791478,0.000000,0.0,0,

## Optional TargetEncoder Experiment

`TargetEncoder` is intentionally kept out of the default prepared outputs. It can be useful for comparison, but it must remain a controlled experiment fit on training data only.


In [10]:
RUN_TARGET_ENCODER_EXPERIMENT = False

if RUN_TARGET_ENCODER_EXPERIMENT:
    from sklearn.preprocessing import TargetEncoder

    target_encoder = TargetEncoder(random_state=42)
    target_encoded_example = target_encoder.fit_transform(
        linear_train_source[LINEAR_CATEGORICAL_COLUMNS],
        linear_ready_train_y
    )
    display(target_encoded_example.head())
else:
    display(
        Markdown(
            "TargetEncoder stays optional here. The default notebook outputs remain `tree_ready` and `linear_ready`, "
            "with no target-encoded features mixed into the baseline prepared datasets."
        )
    )


TargetEncoder stays optional here. The default notebook outputs remain `tree_ready` and `linear_ready`, with no target-encoded features mixed into the baseline prepared datasets.

## Preparation Summary

The table below summarizes the main preparation choices, their rationale, and which prepared branch uses them.


In [11]:
prep_choice_df = pd.DataFrame(
    [
        {
            "prep_decision": "Replace `?` with missing and create missingness flags",
            "reason": "The diagnosis notebook showed meaningful missing-like counts in workclass, occupation, and native.country.",
            "tree_ready": "Yes",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Remove exact duplicate train rows with identical predictors and label",
            "reason": f"{exact_duplicate_count} train rows were pure duplicates excluding Id and can be safely removed from the modeling table.",
            "tree_ready": "Yes",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Preserve conflicting duplicate feature groups",
            "reason": "Conflicting duplicates may represent label noise or unresolved ambiguity, so they are flagged instead of silently dropped.",
            "tree_ready": "Yes",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Keep categorical features natively for the main branch",
            "reason": "Tree boosters can make direct use of categorical structure and do not require normalization.",
            "tree_ready": "Yes",
            "linear_ready": "No"
        },
        {
            "prep_decision": "Use infrequent-aware one-hot encoding for the comparison branch",
            "reason": "This stabilizes linear models when categories like native.country have many rare levels.",
            "tree_ready": "No",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Use log companions and workload flags instead of hard outlier removal",
            "reason": "Hours, capital gains, capital losses, and fnlwgt are skewed or extreme, but mostly plausible enough to preserve.",
            "tree_ready": "Yes",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Separate masters and doctorate in grouped education",
            "reason": "Terminal advanced degrees may carry distinct labor-market signal and should not be merged into one broad postgraduate bucket.",
            "tree_ready": "Yes",
            "linear_ready": "Yes"
        },
        {
            "prep_decision": "Treat TargetEncoder as optional only",
            "reason": "Useful for controlled comparison, but too leakage-sensitive to place in the default prepared output.",
            "tree_ready": "No",
            "linear_ready": "Experimental only"
        }
    ]
)

output_inventory_df = pd.DataFrame(
    [
        {"artifact": "tree_ready_train_X", "rows": len(tree_ready_train_X), "columns": tree_ready_train_X.shape[1]},
        {"artifact": "tree_ready_test_X", "rows": len(tree_ready_test_X), "columns": tree_ready_test_X.shape[1]},
        {"artifact": "linear_ready_train_X", "rows": len(linear_ready_train_X), "columns": linear_ready_train_X.shape[1]},
        {"artifact": "linear_ready_test_X", "rows": len(linear_ready_test_X), "columns": linear_ready_test_X.shape[1]},
        {"artifact": "train_sensitivity_no_conflicts", "rows": len(train_sensitivity_no_conflicts), "columns": train_sensitivity_no_conflicts.shape[1]}
    ]
)

display(prep_choice_df)
display(output_inventory_df)

import json

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared" / "adult_income"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

feature_engineered_train_export = train_features.reset_index(drop=True)
feature_engineered_test_export = test_features.reset_index(drop=True)
tree_ready_train_export = tree_ready_train_X.reset_index(drop=True)
tree_ready_test_export = tree_ready_test_X.reset_index(drop=True)
linear_source_train_export = linear_train_source.reset_index(drop=True)
linear_source_test_export = linear_test_source.reset_index(drop=True)
linear_ready_train_export = linear_ready_train_X.reset_index(drop=True)
linear_ready_test_export = linear_ready_test_X.reset_index(drop=True)

shared_categorical_columns = TREE_CATEGORICAL_COLUMNS.copy()
shared_numeric_columns = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week",
    "workclass_missing",
    "occupation_missing",
    "native_country_missing",
    "log1p_fnlwgt",
    "capital_gain_positive",
    "capital_loss_positive",
    "log1p_capital_gain",
    "log1p_capital_loss",
    "hours_ge_50",
    "hours_ge_60",
    "hours_ge_80"
]
shared_feature_columns = shared_numeric_columns + shared_categorical_columns


def make_shared_prepared_export(frame: pd.DataFrame, include_target: bool) -> pd.DataFrame:
    export_frame = frame[[ID_COL] + shared_feature_columns + ([TARGET_COL] if include_target else [])].copy()
    for column in shared_categorical_columns:
        export_frame[column] = export_frame[column].astype("string").fillna("__MISSING__")
    return export_frame


train_prepared_export = make_shared_prepared_export(train_features, include_target=True)
test_prepared_export = make_shared_prepared_export(test_features, include_target=False)
sample_submission_export = pd.read_csv(DATA_DIR / "sample_submission.csv")

feature_engineered_train_export.to_pickle(PREPARED_DIR / "feature_engineered_train.pkl")
feature_engineered_test_export.to_pickle(PREPARED_DIR / "feature_engineered_test.pkl")
tree_ready_train_export.to_pickle(PREPARED_DIR / "tree_ready_train_X.pkl")
tree_ready_test_export.to_pickle(PREPARED_DIR / "tree_ready_test_X.pkl")
linear_source_train_export.to_pickle(PREPARED_DIR / "linear_source_train.pkl")
linear_source_test_export.to_pickle(PREPARED_DIR / "linear_source_test.pkl")
linear_ready_train_export.to_pickle(PREPARED_DIR / "linear_ready_train_X.pkl")
linear_ready_test_export.to_pickle(PREPARED_DIR / "linear_ready_test_X.pkl")
train_prepared_export.to_csv(PREPARED_DIR / "train_prepared.csv", index=False)
test_prepared_export.to_csv(PREPARED_DIR / "test_prepared.csv", index=False)
sample_submission_export.to_csv(PREPARED_DIR / "sample_submission.csv", index=False)

train_target_export_df = pd.DataFrame(
    {
        ID_COL: feature_engineered_train_export[ID_COL],
        TARGET_COL: feature_engineered_train_export[TARGET_COL],
        "income_binary": tree_ready_train_y.reset_index(drop=True),
        "is_conflicting_duplicate_features": feature_engineered_train_export["is_conflicting_duplicate_features"].astype(bool)
    }
)
train_target_export_df.to_csv(PREPARED_DIR / "train_target.csv", index=False)

test_reference_export_df = feature_engineered_test_export[[ID_COL]].copy()
test_reference_export_df.to_csv(PREPARED_DIR / "test_reference.csv", index=False)

removed_exact_duplicates_df.reset_index(drop=True).to_csv(PREPARED_DIR / "removed_exact_duplicates.csv", index=False)
conflicting_duplicate_rows_df = train_model.loc[
    train_model["is_conflicting_duplicate_features"],
    [ID_COL] + feature_only_columns + [TARGET_COL, "is_conflicting_duplicate_features"]
].reset_index(drop=True)
conflicting_duplicate_rows_df.to_csv(PREPARED_DIR / "conflicting_duplicate_rows.csv", index=False)

export_manifest = {
    "prepared_dir": str(PREPARED_DIR.relative_to(PROJECT_ROOT)),
    "artifacts": {
        "feature_engineered_train": "feature_engineered_train.pkl",
        "feature_engineered_test": "feature_engineered_test.pkl",
        "tree_ready_train_X": "tree_ready_train_X.pkl",
        "tree_ready_test_X": "tree_ready_test_X.pkl",
        "linear_source_train": "linear_source_train.pkl",
        "linear_source_test": "linear_source_test.pkl",
        "linear_ready_train_X": "linear_ready_train_X.pkl",
        "linear_ready_test_X": "linear_ready_test_X.pkl",
        "train_prepared_csv": "train_prepared.csv",
        "test_prepared_csv": "test_prepared.csv",
        "sample_submission": "sample_submission.csv",
        "train_target": "train_target.csv",
        "test_reference": "test_reference.csv",
        "removed_exact_duplicates": "removed_exact_duplicates.csv",
        "conflicting_duplicate_rows": "conflicting_duplicate_rows.csv"
    },
    "recommended_usage": {
        "xgboost": ["tree_ready_train_X.pkl", "tree_ready_test_X.pkl", "train_target.csv", "test_reference.csv"],
        "knn": ["linear_source_train.pkl", "linear_source_test.pkl", "train_target.csv", "test_reference.csv"],
        "neural_network": ["linear_source_train.pkl", "linear_source_test.pkl", "train_target.csv", "test_reference.csv"],
        "precomputed_linear_baseline": ["linear_ready_train_X.pkl", "linear_ready_test_X.pkl", "train_target.csv", "test_reference.csv"],
        "competition_style_pipeline": ["train_prepared.csv", "test_prepared.csv", "sample_submission.csv"]
    },
    "notes": [
        "linear_source_* should be preferred for fold-safe scaling and encoding inside future cross-validation pipelines.",
        "tree_ready_* preserves categorical dtypes through pickle export for tree-based models.",
        "train_prepared.csv and test_prepared.csv provide a familiar single-train single-test interface with aligned feature columns.",
        "Exact duplicate training rows with identical predictors and target were removed before export; conflicting duplicate feature groups were preserved and flagged."
    ]
}
(PREPARED_DIR / "manifest.json").write_text(json.dumps(export_manifest, indent=2))

saved_artifacts_df = pd.DataFrame(
    [
        {"artifact": name, "path": str(PREPARED_DIR / relative_path)}
        for name, relative_path in export_manifest["artifacts"].items()
    ]
)

display(saved_artifacts_df)
display(Markdown(f"Prepared datasets exported to `{PREPARED_DIR.relative_to(PROJECT_ROOT)}`."))


,prep_decision,reason,tree_ready,linear_ready
0,Replace `?` with missing and create missingness flags,"The diagnosis notebook showed meaningful missing-like counts in workclass, occupation, and native.country.",Yes,Yes
1,Remove exact duplicate train rows with identical predictors and label,24 train rows were pure duplicates excluding Id and can be safely removed from the modeling table.,Yes,Yes
2,Preserve conflicting duplicate feature groups,"Conflicting duplicates may represent label noise or unresolved ambiguity, so they are flagged instead of silently dropped.",Yes,Yes
3,Keep categorical features natively for the main branch,Tree boosters can make direct use of categorical structure and do not require normalization.,Yes,No
4,Use infrequent-aware one-hot encoding for the comparison branch,This stabilizes linear models when categories like native.country have many rare levels.,No,Yes
5,Use log companions and workload flags instead of hard outlier removal,"Hours, capital gains, capital losses, and fnlwgt are skewed or extreme, but mostly plausible enough to preserve.",Yes,Yes
6,Separate masters and doctorate in grouped education,Terminal advanced degrees may carry distinct labor-market signal and should not be merged into one broad postgraduate bucket.,Yes,Yes
7,Treat TargetEncoder as optional only,"Useful for controlled comparison, but too leakage-sensitive to place in the default prepared output.",No,Experimental only


,artifact,rows,columns
0,tree_ready_train_X,32536,27
1,tree_ready_test_X,16280,27
2,linear_ready_train_X,32536,128
3,linear_ready_test_X,16280,128
4,train_sensitivity_no_conflicts,32534,21
